# Week 03: Regression, classification, and baselines

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Generate a regression problem

The target is continuous. Generate data from a known linear mechanism plus noise, then hold out test rows before fitting.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge,LogisticRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error,accuracy_score
X=rng.normal(size=(240,3)); y=3*X[:,0]-2*X[:,1]+rng.normal(0,.5,240)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEED)
baseline=DummyRegressor().fit(Xtr,ytr); regression=Ridge(alpha=1).fit(Xtr,ytr)
print("Mean baseline MAE:",mean_absolute_error(yte,baseline.predict(Xte)))
print("Ridge MAE:",mean_absolute_error(yte,regression.predict(Xte)))


**Checkpoint.** MAE is in target units. Predicting the training mean is a meaningful baseline, even when the fitted model performs better.


## 3. Change the question to classification

Define whether the response exceeds zero. This changes the prediction target and metric; it does not turn a regression output into a calibrated probability.


In [ ]:
labels=(y>0).astype(int)
Atr,Ate,btr,bte=train_test_split(X,labels,test_size=.25,stratify=labels,random_state=SEED)
classifier=LogisticRegression(random_state=SEED).fit(Atr,btr)
probability=classifier.predict_proba(Ate)[:,1]
summary={"ridge_test_mae":float(mean_absolute_error(yte,regression.predict(Xte))),
         "classification_accuracy":float(accuracy_score(bte,probability>=.5))}
print(summary)


**Checkpoint.** Describe the event represented by class 1 and why accuracy and MAE answer different questions.


## 4. Plot predicted versus observed

A diagonal reference line shows perfect predictions. Residual scatter conveys error structure.


In [ ]:
prediction=regression.predict(Xte)
plt.scatter(yte,prediction); limits=[min(yte),max(yte)]
plt.plot(limits,limits,"k--"); plt.xlabel("Observed synthetic response")
plt.ylabel("Predicted response"); plt.show()
print("Ridge coefficients:",regression.coef_)


**Checkpoint.** Compare learned coefficients with the generating coefficients [3,-2,0]. Differences reflect sample size, noise, and regularization.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":3,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-03-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)
